# Datathon CDIA 2026 — Pipeline de Dados
## Raio-X do Mercado de Trabalho e Vagas Industriais em Santa Catarina

**Pergunta central de negócio**

> *"Quais regiões e setores industriais de SC enfrentam apagão de mão de obra ou
> expansão acelerada, e como renda e escolaridade explicam esses gargalos?"*

Este notebook implementa o **pipeline completo** (Entregável 1):

1. Ingestão e validação das bases
2. Limpeza e filtragem (exclusões MOV × EX)
3. Cruzamento (join) CAGED × municípios
4. Métricas de negócio (saldo, saldo por 10 mil hab., turnover)
5. Agregações territoriais, setoriais e socioeconômicas
6. Enriquecimento com IA — agrupamento de CBOs em macrofamílias
7. Exportação das tabelas-base para o dashboard

Todas as tabelas finais são gravadas em `outputs/` e consumidas pelo dashboard (Entregável 2).


## 0. Configuração e dependências

```bash
pip install -r requirements.txt   # pandas, pyarrow, openpyxl (+ streamlit/pydeck/plotly p/ dashboard)
```
A etapa de IA (seção 8) **não chama API ao executar o notebook**: ela carrega a classificação
gerada pelo LLM e versionada em `ia/cbo_macrofamilias.csv`. Para refazê-la via API, veja
`ia/classificar_cbos.py` (requer `ANTHROPIC_API_KEY`).


In [1]:
import os
import unicodedata
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Caminhos (ajuste se executar de outro diretório)
BASE = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
DATASET = os.path.join(BASE, "dataset")
OUTPUTS = os.path.join(BASE, "outputs")
os.makedirs(OUTPUTS, exist_ok=True)

CAGED_PARQUET = os.path.join(DATASET, "Novo_Caged_MOV_Limpo.parquet")
MUNIC_CSV     = os.path.join(DATASET, "Municipios_SC_limpo.csv")
DICIONARIO    = os.path.join(DATASET, "Dicionario_de_dados.xlsx")
IBGE_MUNIC    = os.path.join(DATASET, "ibge", "municipios_sc.json")   # município -> mesorregião (API IBGE)
CBO_IA        = os.path.join(BASE, "ia", "cbo_macrofamilias.csv")     # classificação gerada por LLM
CBO_ENTRADA   = os.path.join(BASE, "ia", "cbos_para_classificar.csv") # CBOs + títulos enviados ao LLM

print("dataset:", DATASET)
print("outputs:", OUTPUTS)

dataset: ../dataset
outputs: ../outputs


## 1. Ingestão e validação

O CAGED está em **Parquet (zstd)** — ~42 MB, 3.397.416 movimentações, 30 colunas,
somente Santa Catarina (uf=42). Lemos os dicionários para traduzir os códigos.


In [2]:
# --- CAGED (movimentações) ---
caged = pd.read_parquet(CAGED_PARQUET)
print("CAGED:", caged.shape)

# --- Municípios (demografia / socioeconômico) ---
munic = pd.read_csv(MUNIC_CSV)
print("Municípios:", munic.shape)

# --- Dicionário de dados: cada aba mapeia Código -> Descrição ---
xls = pd.ExcelFile(DICIONARIO)
def tabela_dic(aba):
    d = pd.read_excel(DICIONARIO, sheet_name=aba, dtype=str)
    d.columns = [c.strip().lower() for c in d.columns]
    return dict(zip(d["código"].astype(str).str.strip(),
                    d["descrição"].astype(str).str.strip()))

MAP_SECAO   = tabela_dic("seção")
MAP_INSTR   = tabela_dic("graudeinstrução")
MAP_TIPOMOV = tabela_dic("tipomovimentação")
MAP_MUNIC   = tabela_dic("município")
print("Abas do dicionário:", xls.sheet_names)

CAGED: (3397416, 30)
Municípios: (296, 15)


Abas do dicionário: ['Layout', 'região', 'uf', 'município', 'seção', 'subclasse', 'categoria', 'cbo2002ocupação', 'graudeinstrução', 'raçacor', 'sexo', 'tipoempregador', 'tipoestabelecimento', 'tipomovimentação', 'tipodedeficiência', 'indtrabintermitente', 'indtrabparcial', 'tamestabjan', 'indicadoraprendiz', 'origemdainformação', 'indicadordeexclusão', 'indicadordeforadoprazo', 'unidadesaláriocódigo']


In [3]:
# Validação de schema mínima
esperadas = {"competênciamov","uf","município","seção","saldomovimentação",
             "cbo2002ocupação","graudeinstrução","salário","tipomovimentação"}
faltando = esperadas - set(caged.columns)
assert not faltando, f"Colunas ausentes no CAGED: {faltando}"
print("Schema OK. UFs presentes:", caged["uf"].unique())
print("Competências de declaração (competênciadec):", caged["competênciadec"].nunique(),
      "meses, de", caged["competênciadec"].min(), "a", caged["competênciadec"].max())
fora = (caged["indicadordeforadoprazo"] == "1").mean()
print(f"Declarações fora do prazo (competênciamov anterior a set/2025): {fora:.2%}")

Schema OK. UFs presentes: <ArrowStringArray>
['42']
Length: 1, dtype: str
Competências de declaração (competênciadec): 12 meses, de 202509 a 202608
Declarações fora do prazo (competênciamov anterior a set/2025): 1.09%


## 2. Limpeza, tipagem e filtragem (exclusões MOV × EX)

No Parquet todas as colunas vêm como **texto** (preserva fidelidade). Aqui convertemos
os tipos e aplicamos o filtro de exclusões exigido pelo guia.

**Sobre o filtro MOV × EX (premissa explícita):** o Novo CAGED é publicado em três arquivos —
MOV (movimentações no prazo), FOR (fora do prazo) e EXC (exclusões). Recebemos apenas a base
**MOV já limpa** (`Novo_Caged_MOV_Limpo`), que não traz a coluna `indicadordeexclusão` nem o
arquivo EXC. Portanto: (1) mantemos uma verificação defensiva — se a coluna existir, removemos
`=1`; (2) declaramos como **limite** que exclusões posteriores não puderam ser abatidas.

Outros tratamentos:
- **Salário:** 1% dos registros têm salário 0 e o máximo chega a R$ 3,4 milhões (erros de
  declaração). Para médias salariais consideramos apenas a faixa **R$ 500 – R$ 50.000**; os
  registros fora dela continuam contando no saldo, só não entram no salário médio.
- **Escolaridade:** o código CAGED não é ordinal (80 = pós-graduação, 99 = não identificado),
  então a média dos códigos não faz sentido. Convertemos para **anos de estudo** aproximados
  e criamos o indicador **% com ensino superior completo**.


In [4]:
df = caged.copy()

# Colunas numéricas (salário usa vírgula decimal e ponto de milhar)
def to_float_br(s):
    return pd.to_numeric(
        s.astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce")

df["saldo"]          = pd.to_numeric(df["saldomovimentação"], errors="coerce")
df["salario"]        = to_float_br(df["salário"])
df["salario_fixo"]   = to_float_br(df.get("valorsaláriofixo", pd.Series(index=df.index)))
df["idade"]          = pd.to_numeric(df["idade"], errors="coerce")
df["grau_cod"]       = pd.to_numeric(df["graudeinstrução"], errors="coerce")
df["mun_cod"]        = df["município"].astype(str).str.strip()
df["cbo"]            = df["cbo2002ocupação"].astype(str).str.strip().str.zfill(6)

# Salário válido para médias (fora da faixa vira NaN só na coluna de salário)
df.loc[~df["salario"].between(500, 50_000), "salario"] = np.nan

# Escolaridade -> anos de estudo (aprox.) e flag de superior completo
ANOS_ESTUDO = {1: 0, 2: 2, 3: 5, 4: 7, 5: 9, 6: 10, 7: 12, 8: 13.5,
               9: 16, 80: 17, 10: 18, 11: 22}            # 99 = não identificado -> NaN
df["anos_estudo"] = df["grau_cod"].map(ANOS_ESTUDO)
df["superior"]    = df["grau_cod"].isin([9, 10, 11, 80]).astype(float)
df.loc[df["grau_cod"] == 99, "superior"] = np.nan

# Filtro de exclusões (defensivo)
if "indicadordeexclusão" in df.columns:
    antes = len(df)
    df = df[df["indicadordeexclusão"].astype(str).str.strip() != "1"].copy()
    print(f"Exclusões removidas: {antes - len(df)}")
else:
    print("Sem coluna de exclusão (base já limpa na origem).")

# Descartar artefatos de índice
df = df.drop(columns=[c for c in ["Unnamed: 0.1", "Unnamed: 0"] if c in df.columns])

# Traduções legíveis
df["setor"]      = df["seção"].map(MAP_SECAO).fillna("Não identificado")
df["grau_instr"] = df["graudeinstrução"].astype(str).map(MAP_INSTR).fillna("Não Identificado")
df["tipo_mov"]   = df["tipomovimentação"].astype(str).map(MAP_TIPOMOV).fillna("Não Identificado")

df["admissao"]     = (df["saldo"] > 0).astype(int)
df["desligamento"] = (df["saldo"] < 0).astype(int)
# Motivo do desligamento: "a pedido" (40) = trabalhador saiu por vontade própria — sinal de
# mercado aquecido / dificuldade de reter (indício de apagão); demissão (31, 32) = iniciativa da empresa
df["a_pedido"] = (df["tipomovimentação"].astype(str) == "40").astype(int)
df["demissao"] = df["tipomovimentação"].astype(str).isin(["31", "32"]).astype(int)

print("Linhas após limpeza:", len(df))
df[["competênciamov","mun_cod","setor","saldo","salario","grau_instr"]].head()

Sem coluna de exclusão (base já limpa na origem).


Linhas após limpeza: 3397416


,competênciamov,mun_cod,setor,saldo,salario,grau_instr
0,202509,420890,Construção,1,2018.00,Médio Completo
1,202509,421605,"Comércio, Reparação de Veículos Automotores e ...",-1,1693.35,Médio Completo
2,202509,420820,Construção,1,2556.00,Médio Incompleto
3,202509,420820,"Comércio, Reparação de Veículos Automotores e ...",-1,3181.34,Superior Completo
4,202509,421555,Indústrias de Transformação,-1,1650.00,Superior Incompleto


## 3. Cruzamento CAGED × municípios

Chave: código do município. No `Municipios_SC_limpo.csv` o código IBGE tem 7 dígitos;
no CAGED, 6 (sem o dígito verificador). Alinhamos pelos **6 primeiros dígitos**.


In [5]:
# Normaliza nomes de colunas do arquivo de municípios
munic.columns = [c.strip() for c in munic.columns]

def achar_col(cols, *chaves):
    for c in cols:
        low = c.lower()
        if all(k in low for k in chaves):
            return c
    return None

col_codigo = achar_col(munic.columns, "código")
col_nome   = achar_col(munic.columns, "município")
# população estimada 2026 (mais próxima do período analisado, 2025-26); fallback: censo 2022
col_pop    = achar_col(munic.columns, "população", "estimada") or achar_col(munic.columns, "população", "último censo")
col_idhm   = achar_col(munic.columns, "idhm")
col_pib    = achar_col(munic.columns, "pib per capita") or achar_col(munic.columns, "pib")
col_escol  = achar_col(munic.columns, "escolariza")

print("Mapeadas:", dict(codigo=col_codigo, nome=col_nome, pop=col_pop,
                         idhm=col_idhm, pib=col_pib, escol=col_escol))

mun = pd.DataFrame({
    "mun_cod":   munic[col_codigo].astype(str).str.replace(r"\D", "", regex=True).str[:6],
    "municipio": munic[col_nome].astype(str).str.strip(),
})
def num(col):
    # este CSV usa PONTO decimal (ex.: IDHM 0.694, PIB 44644.04) -> conversão direta
    if col is None: return np.nan
    return pd.to_numeric(munic[col], errors="coerce")

mun["populacao"]     = num(col_pop)
mun["idhm"]          = num(col_idhm)
mun["pib_per_capita"]= num(col_pib)
mun["escolarizacao"] = num(col_escol)
# o CSV termina com linhas de notas/fontes (HTML) -> só ficam linhas com código numérico
mun = mun[mun["mun_cod"].str.fullmatch(r"\d{6}")].drop_duplicates("mun_cod")

# Mesorregião IBGE (6 regiões de SC) — base do mapa do dashboard
import json
with open(IBGE_MUNIC, encoding="utf-8") as f:
    ibge = json.load(f)
meso = pd.DataFrame({
    "mun_cod":        [str(m["id"])[:6] for m in ibge],
    "cod_meso":       [m["microrregiao"]["mesorregiao"]["id"] for m in ibge],
    "mesorregiao":    [m["microrregiao"]["mesorregiao"]["nome"] for m in ibge],
    "microrregiao":   [m["microrregiao"]["nome"] for m in ibge],
})
# inner join com a lista oficial do IBGE: descarta linhas de rodapé que não são municípios
antes = len(mun)
mun = mun.merge(meso, on="mun_cod", how="inner")
print(f"Linhas descartadas (não-municípios): {antes - len(mun)}")
assert len(mun) == 295, "SC tem 295 municípios"
print("Municípios normalizados:", mun.shape)
print(mun.groupby("mesorregiao")["populacao"].agg(["size", "sum"]).rename(columns={"size": "municipios"}))
mun.head()

Mapeadas: {'codigo': 'Código [-]', 'nome': 'Município [-]', 'pop': 'População estimada - pessoas [2026]', 'idhm': 'IDHM <span>Índice de desenvolvimento humano municipal</span> [2010]', 'pib': 'PIB per capita - R$ [2023]', 'escol': 'Escolarização <span>6 a 14 anos</span> - % [2022]'}
Linhas descartadas (não-municípios): 1
Municípios normalizados: (295, 9)
                      municipios        sum
mesorregiao                                
Grande Florianópolis          21  1490562.0
Norte Catarinense             26  1573204.0
Oeste Catarinense            118  1447089.0
Serrana                       30   439817.0
Sul Catarinense               46  1163640.0
Vale do Itajaí                54  2198447.0


,mun_cod,municipio,populacao,idhm,pib_per_capita,escolarizacao,cod_meso,mesorregiao,microrregiao
0,420005,Abdon Batista,2628.0,0.694,44644.04,100.00,4203,Serrana,Curitibanos
1,420010,Abelardo Luz,17797.0,0.696,62434.34,99.69,4201,Oeste Catarinense,Xanxerê
2,420020,Agrolândia,11798.0,0.725,45141.98,99.47,4204,Vale do Itajaí,Ituporanga
3,420030,Agronômica,6453.0,0.741,46144.62,98.98,4204,Vale do Itajaí,Rio do Sul
4,420040,Água Doce,6547.0,0.698,114906.54,99.03,4201,Oeste Catarinense,Joaçaba


In [6]:
dfm = df.merge(mun, on="mun_cod", how="left")
sem_match = dfm["municipio"].isna().mean()
print(f"% movimentações sem município correspondente: {sem_match:.2%}")
dfm[["mun_cod","municipio","populacao","setor","saldo"]].head()

% movimentações sem município correspondente: 0.00%


,mun_cod,municipio,populacao,setor,saldo
0,420890,Jaraguá do Sul,203204.0,Construção,1
1,421605,São Cristóvão do Sul,6586.0,"Comércio, Reparação de Veículos Automotores e ...",-1
2,420820,Itajaí,302247.0,Construção,1
3,420820,Itajaí,302247.0,"Comércio, Reparação de Veículos Automotores e ...",-1
4,421555,Santa Helena,2480.0,Indústrias de Transformação,-1


## 4. Métricas de negócio

- **Saldo** = Σ(saldomovimentação) = admissões − desligamentos.
- **Saldo por 10.000 habitantes** = saldo / população × 10.000 (normalização exigida pelo guia).
- **Turnover (rotatividade)** = (admissões + desligamentos) / 2 ÷ estoque aproximado.
  Como não temos o estoque de vínculos, usamos a **taxa de giro** = (adm+desl) por 10 mil hab,
  que diferencia "apagão" (giro alto, saldo baixo) de "expansão" (saldo alto).


In [7]:
def agrega(grupo_cols, nome_pop="populacao"):
    g = dfm.groupby(grupo_cols, dropna=False).agg(
        admissoes=("admissao", "sum"),
        desligamentos=("desligamento", "sum"),
        saldo=("saldo", "sum"),
        movimentacoes=("saldo", "size"),
        salario_medio=("salario", "mean"),
        salario_mediano=("salario", "median"),
        anos_estudo=("anos_estudo", "mean"),
        pct_superior=("superior", "mean"),
        idade_media=("idade", "mean"),
    ).reset_index()
    g["giro"] = g["admissoes"] + g["desligamentos"]
    # taxa de reposição: desligamentos para cada admissão (≈1 = só repõe quem sai -> rotatividade)
    g["reposicao"] = g["desligamentos"] / g["admissoes"].where(g["admissoes"] > 0)
    return g

# Base populacional por município (para normalizar)
pop_mun = mun.set_index("mun_cod")["populacao"]

## 5. Eixo territorial — municípios em expansão vs. apagão

In [8]:
terr = agrega(["mun_cod", "municipio"])
terr = terr.merge(mun[["mun_cod","populacao","idhm","pib_per_capita","escolarizacao"]],
                  on="mun_cod", how="left")
terr["saldo_por_10k"] = terr["saldo"] / terr["populacao"] * 10_000
terr["giro_por_10k"]  = terr["giro"]  / terr["populacao"] * 10_000

print("TOP 10 EXPANSÃO (saldo por 10 mil hab.):")
display_cols = ["municipio","saldo","saldo_por_10k","giro_por_10k","salario_medio","anos_estudo"]
print(terr.sort_values("saldo_por_10k", ascending=False)[display_cols].head(10).to_string(index=False))

print("\nTOP 10 RETRAÇÃO / APAGÃO (menor saldo por 10 mil hab.):")
print(terr.sort_values("saldo_por_10k")[display_cols].head(10).to_string(index=False))

TOP 10 EXPANSÃO (saldo por 10 mil hab.):
                municipio  saldo  saldo_por_10k  giro_por_10k  salario_medio  anos_estudo
   Governador Celso Ramos   1571     843.173036   7924.538429    2158.675500    11.211141
                 Guatambú    581     584.095707   6941.791495    2265.255205     7.106227
Santo Amaro da Imperatriz   1282     416.368951   4447.547905    2377.668098    11.916204
               Navegantes   3514     357.037624   4765.039981    2518.583784    11.182545
      Bom Jardim da Serra    141     355.701312   5315.338042    1926.218328    11.006645
             Treze Tílias    344     352.856703   6021.130372    2443.043365    10.887394
             Nova Erechim    190     342.157392   2553.574644    2398.309270    10.476375
                  Vargeão    123     329.847144   5199.785465    2437.136668    10.607014
           Pedras Grandes    129     295.126973   4358.270419    2395.816157    11.401837
           Otacílio Costa    481     267.742833   4283.3286

In [9]:
# Agregação por MESORREGIÃO (6 regiões de SC) — nível principal do mapa do dashboard
pop_meso = mun.groupby("mesorregiao")["populacao"].sum()
reg = agrega(["cod_meso", "mesorregiao"])
reg["populacao"]     = reg["mesorregiao"].map(pop_meso)
reg["saldo_por_10k"] = reg["saldo"] / reg["populacao"] * 10_000
reg["giro_por_10k"]  = reg["giro"]  / reg["populacao"] * 10_000
print(reg.sort_values("saldo_por_10k", ascending=False)
        [["mesorregiao","saldo","saldo_por_10k","giro_por_10k","reposicao",
          "salario_medio","anos_estudo","pct_superior"]].round(3).to_string(index=False))

         mesorregiao  saldo  saldo_por_10k  giro_por_10k  reposicao  salario_medio  anos_estudo  pct_superior
Grande Florianópolis  12684         85.095      4546.983      0.963       2500.196       11.725         0.106
   Oeste Catarinense   9360         64.682      3689.324      0.966       2364.376       10.763         0.071
      Vale do Itajaí  10454         47.552      4608.371      0.980       2528.477       11.403         0.078
   Norte Catarinense   6097         38.755      4020.108      0.981       2481.829       11.528         0.085
             Serrana   1686         38.334      3109.930      0.976       2195.462       10.841         0.056
     Sul Catarinense   1201         10.321      3466.974      0.994       2384.408       11.516         0.075


## 6. Eixo setorial — setores industriais que lideram

Foco industrial = seções **B** (Extrativas), **C** (Transformação), **D** (Eletricidade/Gás),
**F** (Construção). Analisamos todos e sinalizamos os industriais.


In [10]:
SETORES_INDUSTRIAIS = {"B", "C", "D", "F"}
setor = agrega(["seção", "setor"])
setor["industrial"] = setor["seção"].isin(SETORES_INDUSTRIAIS)
setor = setor.sort_values("saldo", ascending=False)
print(setor[["setor","saldo","admissoes","desligamentos","giro",
             "salario_medio","anos_estudo","industrial"]].to_string(index=False))

                                                            setor  saldo  admissoes  desligamentos   giro  salario_medio  anos_estudo  industrial
             Atividades Administrativas e Serviços Complementares  11416     233612         222196 455808    2136.804111    11.342920       False
                 Atividades Profissionais, Científicas e Técnicas   6877      56443          49566 106009    2731.275250    12.478110       False
       Comércio, Reparação de Veículos Automotores e Motocicletas   5655     398899         393244 792143    2289.483383    11.535003       False
                                Transporte, Armazenagem e Correio   5290     111905         106615 218520    2524.657005    11.310379       False
                                  Saúde Humana e Serviços Sociais   4603      63262          58659 121921    2631.720722    12.617584       False
                                                       Construção   1737     137306         135569 272875    2754.229465    

In [11]:
# Cruzamento território × setor (onde cada setor cresce/encolhe)
terr_setor = agrega(["mun_cod","municipio","seção","setor"])
terr_setor = terr_setor.merge(pop_mun.rename("populacao"), on="mun_cod", how="left")
terr_setor["saldo_por_10k"] = terr_setor["saldo"] / terr_setor["populacao"] * 10_000
# Exemplo: maiores polos industriais por saldo
ind = terr_setor[terr_setor["seção"].isin(SETORES_INDUSTRIAIS)]
print("Maiores saldos industriais (município × setor):")
print(ind.sort_values("saldo", ascending=False)
        [["municipio","setor","saldo","salario_medio","anos_estudo"]].head(15).to_string(index=False))

Maiores saldos industriais (município × setor):
         municipio                       setor  saldo  salario_medio  anos_estudo
        Navegantes Indústrias de Transformação   2377    2387.885769    10.824528
         Joinville Indústrias de Transformação   1479    2773.040891    11.519775
           Chapecó Indústrias de Transformação   1296    2486.248674     9.585045
           Caçador                  Construção    736    3180.329558    10.777958
          Guatambú Indústrias de Transformação    561    2298.617007     6.485883
           Brusque                  Construção    531    2803.705670    10.644716
            Itajaí Indústrias de Transformação    508    2658.931698    11.261894
         Canoinhas                  Construção    441    1891.010391    10.828817
Balneário Camboriú                  Construção    341    3028.645613    11.034640
        Navegantes                  Construção    323    3123.974064    10.809899
          Camboriú                  Construção    

## 7. Eixo socioeconômico — renda e escolaridade explicam os gargalos?

Comparamos o perfil de **admitidos vs. desligados** por escolaridade e salário, e cruzamos
o saldo setorial/territorial com renda e escolaridade para apontar a causa-raiz:
desatratividade salarial, escolaridade não atendida ou compressão salarial.


In [12]:
# Perfil admitido vs desligado por escolaridade
perfil = (dfm.assign(mov=np.where(dfm["saldo"] > 0, "Admitido", "Desligado"))
            .groupby(["mov","grau_instr"], dropna=False)
            .agg(n=("saldo","size"), salario_medio=("salario","mean"))
            .reset_index())
print(perfil.to_string(index=False))

# Salário médio por grau de instrução (há compressão salarial?)
sal_grau = (dfm.groupby("grau_cod")
              .agg(grau=("grau_instr","first"),
                   salario_medio=("salario","mean"),
                   n=("salario","size"))
              .reset_index().sort_values("grau_cod"))
print("\nSalário médio por grau de instrução:")
print(sal_grau[["grau","salario_medio","n"]].to_string(index=False))

      mov              grau_instr       n  salario_medio
 Admitido 5ª Completo Fundamental   23242    2074.328691
 Admitido     6ª a 9ª Fundamental   76510    2102.941807
 Admitido              Analfabeto   15539    2206.174211
 Admitido       Até 5ª Incompleto   27075    2199.750530
 Admitido               Doutorado     721    6984.643670
 Admitido    Fundamental Completo  143155    2206.855751
 Admitido                Mestrado    1912    5990.205818
 Admitido          Médio Completo 1031476    2304.315558
 Admitido        Médio Incompleto  178453    2000.054356
 Admitido  Pós-Graduação completa   16180    5467.591842
 Admitido       Superior Completo  119642    3813.647556
 Admitido     Superior Incompleto   85544    2569.588943
Desligado 5ª Completo Fundamental   23422    2170.895856
Desligado     6ª a 9ª Fundamental   75847    2189.811365
Desligado              Analfabeto   15356    2412.796170
Desligado       Até 5ª Incompleto   25418    2254.369178
Desligado               Doutora

## 8. Enriquecimento com IA — agrupamento de CBOs em macrofamílias

**Problema:** o CAGED-SC tem **2.207 CBOs distintos** — impossível analisar ocupação por
ocupação. A estrutura oficial da CBO (1º dígito = Grande Grupo) separa por *nível*, mas não
por *cadeia industrial*: soldador, costureira e magarefe caem todos no mesmo grupo "7/8".
Para responder *"o apagão é operacional ou técnico? em qual indústria?"* precisamos de
macrofamílias pensadas para a indústria catarinense.

**Como fizemos (IA Generativa):**
1. Montamos a lista `código | título` dos 2.207 CBOs (títulos do dicionário oficial; 28 CBOs
   sem título receberam como contexto os títulos da mesma família CBO) → `ia/cbos_para_classificar.csv`.
2. Escrevemos um prompt com **14 macrofamílias fixas** e regras de desempate →
   `ia/prompt_classificacao_cbo.md`. Cada CBO recebe: `macrofamilia`, `nivel`
   (operacional / técnico / superior / gestão) e `industrial` (S/N).
3. Enviamos ao LLM (**Claude Opus 5.5**) em **5 lotes de ~440 CBOs**, exigindo resposta CSV
   com os mesmos códigos e na mesma ordem; cada lote foi validado automaticamente
   (códigos idênticos, valores dentro do domínio).
4. Resultado versionado em `ia/cbo_macrofamilias.csv` — o notebook **não chama API**
   (reprodutível, sem custo, sem depender de rede). Para refazer via API:
   `python ia/classificar_cbos.py`.
5. **Validação:** `python ia/validar_classificacao.py` (cobertura, coerência com o Grande
   Grupo oficial, amostra para revisão humana em `ia/amostra_revisao.csv`).


In [13]:
# 8.1 Carrega a classificação gerada pelo LLM
NOMES_MACRO = {
    "OP_METAL":  "Operacional – Metalmecânica e eletroeletrônica",
    "OP_TEXTIL": "Operacional – Têxtil, confecção e calçados",
    "OP_ALIM":   "Operacional – Alimentos, bebidas e agroindústria",
    "OP_OUTRAS": "Operacional – Outras indústrias de transformação",
    "CONSTR":    "Construção civil e infraestrutura",
    "MANUT":     "Manutenção, reparação e instalação",
    "LOGIST":    "Logística, transporte e armazenagem",
    "TEC_IND":   "Técnicos industriais, de TI e qualidade",
    "ENG_TI":    "Engenharia, TI e ciências (superior)",
    "GESTAO":    "Gestão, supervisão e liderança",
    "ADM":       "Administrativo e escritório",
    "COMERCIO":  "Comércio, vendas e atendimento",
    "SERV":      "Serviços (saúde, educação, alimentação etc.)",
    "AGRO":      "Agropecuária, pesca e florestal",
    "NAO_CLASSIFICADO": "Não classificado",
}
cbo_ia = pd.read_csv(CBO_IA, dtype=str)
cbo_ia["macrofamilia_nome"] = cbo_ia["macrofamilia"].map(NOMES_MACRO)
titulos = pd.read_csv(CBO_ENTRADA, dtype={"cbo": str})[["cbo", "titulo"]]
cbo_ia = cbo_ia.merge(titulos, on="cbo", how="left")
print("CBOs classificados:", len(cbo_ia))
print(cbo_ia["macrofamilia"].value_counts().to_string())

# Exemplos de como o LLM classificou
cbo_ia[cbo_ia["cbo"].isin(["724315","763215","848520","784205","214905","313410","717020","782510"])]

CBOs classificados: 2207
macrofamilia
SERV                537
OP_OUTRAS           251
GESTAO              233
ADM                 153
OP_METAL            151
TEC_IND             149
ENG_TI              138
AGRO                121
OP_TEXTIL           113
LOGIST               96
MANUT                90
CONSTR               73
OP_ALIM              51
COMERCIO             50
NAO_CLASSIFICADO      1


,cbo,macrofamilia,nivel,industrial,macrofamilia_nome,titulo
231,214905,ENG_TI,SUPERIOR,S,"Engenharia, TI e ciências (superior)",Engenheiro de Producao
664,313410,TEC_IND,TECNICO,S,"Técnicos industriais, de TI e qualidade",Tecnico em Instrumentacao
1457,717020,CONSTR,OPERACIONAL,S,Construção civil e infraestrutura,Servente de Obras
1544,724315,OP_METAL,OPERACIONAL,S,Operacional – Metalmecânica e eletroeletrônica,Soldador
1732,763215,OP_TEXTIL,OPERACIONAL,S,"Operacional – Têxtil, confecção e calçados","Costureiro, a Maquina na Confeccao em Serie"
1880,782510,LOGIST,OPERACIONAL,S,"Logística, transporte e armazenagem",Motorista de Caminhao (Rotas Regionais e Inter...
1912,784205,OP_OUTRAS,OPERACIONAL,S,Operacional – Outras indústrias de transformação,Alimentador de Linha de Producao
2096,848520,OP_ALIM,OPERACIONAL,S,"Operacional – Alimentos, bebidas e agroindústria",Magarefe


In [14]:
# 8.2 Junta a classificação às movimentações
dfm = dfm.merge(cbo_ia[["cbo","macrofamilia","macrofamilia_nome","nivel"]]
                  .rename(columns={"nivel": "nivel_ocup"}),
                on="cbo", how="left")
cobertura = dfm["macrofamilia"].notna().mean()
print(f"Movimentações com macrofamília: {cobertura:.2%}")
assert cobertura > 0.999

Movimentações com macrofamília: 100.00%


In [15]:
# 8.3 Validação rápida: coerência com a estrutura oficial da CBO (1º dígito = Grande Grupo)
GG_ESPERADO = {"ENG_TI": {"2"}, "TEC_IND": {"3"}, "AGRO": {"6"}, "CONSTR": {"7"},
               "OP_METAL": {"7","8"}, "OP_TEXTIL": {"7","8"}, "OP_ALIM": {"7","8"},
               "OP_OUTRAS": {"7","8"}, "MANUT": {"7","9"}}
gg = cbo_ia["cbo"].str[0]
coerente = [g in GG_ESPERADO.get(m, {g}) for g, m in zip(gg, cbo_ia["macrofamilia"])]
print(f"Coerência com o Grande Grupo oficial: {np.mean(coerente):.1%} dos CBOs")
print("Divergências não são necessariamente erros (ex.: 'Auxiliar de Produção Farmacêutica'"
      " está no GG 5 da CBO, mas é trabalho fabril) — são a lista para revisão humana.")

Coerência com o Grande Grupo oficial: 98.0% dos CBOs
Divergências não são necessariamente erros (ex.: 'Auxiliar de Produção Farmacêutica' está no GG 5 da CBO, mas é trabalho fabril) — são a lista para revisão humana.


In [16]:
# 8.4 Macrofamílias: saldo, rotatividade, renda e escolaridade
macro = agrega(["macrofamilia", "macrofamilia_nome"])
macro["n_cbos"] = macro["macrofamilia"].map(cbo_ia["macrofamilia"].value_counts())
print(macro.sort_values("giro", ascending=False)
        [["macrofamilia_nome","n_cbos","admissoes","desligamentos","saldo","reposicao",
          "salario_medio","anos_estudo","pct_superior"]].round(3).to_string(index=False))

                               macrofamilia_nome  n_cbos  admissoes  desligamentos  saldo  reposicao  salario_medio  anos_estudo  pct_superior
    Serviços (saúde, educação, alimentação etc.)     537     364200         347622  16578      0.954       2288.636       11.497         0.122
                  Comércio, vendas e atendimento      50     292554         275696  16858      0.942       2122.969       11.719         0.056
             Logística, transporte e armazenagem      96     207591         198742   8849      0.957       2428.331       11.162         0.027
Operacional – Outras indústrias de transformação     251     202697         190090  12607      0.938       2203.064       10.690         0.016
                     Administrativo e escritório     153     185041         179669   5372      0.971       2508.893       12.617         0.205
               Construção civil e infraestrutura      73     112609         110236   2373      0.979       2646.682       10.414         0.005

In [17]:
# 8.5 O que a IA revela que o Grande Grupo esconde:
# no baseline, todo o operacional industrial é um bloco só (GG 7/8)
GRANDE_GRUPO_CBO = {"0": "Forças Armadas", "1": "Dirigentes", "2": "Prof. nível superior",
                    "3": "Técnicos nível médio", "4": "Apoio administrativo",
                    "5": "Serviços e comércio", "6": "Agropecuária",
                    "7": "Produção industrial (7)", "8": "Produção industrial (8)",
                    "9": "Manutenção"}
dfm["cbo_grande_grupo"] = dfm["cbo"].str[0].map(GRANDE_GRUPO_CBO)
oper = dfm[dfm["cbo_grande_grupo"].str.startswith("Produção")]
print("Dentro do bloco 'Produção industrial' (GG 7/8), a IA separa:")
print(oper.groupby("macrofamilia_nome")
        .agg(movs=("saldo","size"), saldo=("saldo","sum"), salario_medio=("salario","mean"))
        .sort_values("movs", ascending=False).round(0).to_string())

Dentro do bloco 'Produção industrial' (GG 7/8), a IA separa:
                                                    movs  saldo  salario_medio
macrofamilia_nome                                                             
Operacional – Outras indústrias de transformação  391063  12575         2202.0
Construção civil e infraestrutura                 219766   2242         2655.0
Logística, transporte e armazenagem               215979   1615         2579.0
Operacional – Metalmecânica e eletroeletrônica    126794   -742         2801.0
Operacional – Têxtil, confecção e calçados        122750  -3882         2239.0
Operacional – Alimentos, bebidas e agroindústria   86259   1237         2281.0
Manutenção, reparação e instalação                 24072    140         2457.0
Gestão, supervisão e liderança                     15698  -2876         4345.0
Técnicos industriais, de TI e qualidade             1966     50         2450.0
Serviços (saúde, educação, alimentação etc.)          73     -5       

In [18]:
# 8.6 Macrofamília × mesorregião (onde está o gargalo de cada família ocupacional)
macro_reg = agrega(["cod_meso","mesorregiao","macrofamilia","macrofamilia_nome"])
macro_reg["populacao"] = macro_reg["mesorregiao"].map(pop_meso)
macro_reg["saldo_por_10k"] = macro_reg["saldo"] / macro_reg["populacao"] * 10_000
# Sinais de apagão: muita movimentação + reposição ~1 (contrata só para repor)
sinais = macro_reg[(macro_reg["giro"] > 5000) & (macro_reg["macrofamilia"].str.startswith(("OP_","TEC","ENG","MANUT")))]
print(sinais.sort_values("reposicao", ascending=False)
        [["mesorregiao","macrofamilia_nome","giro","saldo","reposicao","salario_medio","anos_estudo"]]
        .head(15).round(3).to_string(index=False))

         mesorregiao                                macrofamilia_nome  giro  saldo  reposicao  salario_medio  anos_estudo
   Oeste Catarinense       Operacional – Têxtil, confecção e calçados  9308   -516      1.117       1960.534       10.408
   Norte Catarinense       Operacional – Têxtil, confecção e calçados 19041   -941      1.104       2349.008       10.965
     Sul Catarinense   Operacional – Metalmecânica e eletroeletrônica 18375   -901      1.103       2823.303       11.116
      Vale do Itajaí Operacional – Alimentos, bebidas e agroindústria 14083   -687      1.103       2428.500       10.594
   Oeste Catarinense          Técnicos industriais, de TI e qualidade 13599   -647      1.100       3028.462       12.458
   Norte Catarinense Operacional – Alimentos, bebidas e agroindústria  7481   -339      1.095       2394.655       10.882
      Vale do Itajaí             Engenharia, TI e ciências (superior)  5204   -222      1.089       6310.071       14.597
     Sul Catarinense    

## 9. Exportação das tabelas-base (consumidas pelo dashboard)

Gravamos agregações enxutas em `outputs/` — são elas que o dashboard (Entregável 2) carrega.
O **cubo** (`cubo_movimentacoes.parquet`) guarda somas por município × seção CNAE ×
macrofamília × nível × tipo de movimento, permitindo ao dashboard recalcular qualquer recorte
(região, setor, família) sem reler os 3,4 milhões de linhas.


In [19]:
# Cubo de somas (médias são recalculadas no dashboard como soma/contagem)
dfm["tipo"] = np.where(dfm["saldo"] > 0, "ADM", "DESL")
cubo = (dfm.groupby(["mun_cod","seção","macrofamilia","nivel_ocup","tipo"], dropna=False)
          .agg(n=("saldo","size"),
               a_pedido=("a_pedido","sum"),  demissao=("demissao","sum"),
               sal_soma=("salario","sum"),   sal_n=("salario","count"),
               anos_soma=("anos_estudo","sum"), anos_n=("anos_estudo","count"),
               sup_soma=("superior","sum"),  sup_n=("superior","count"))
          .reset_index())
print("Cubo:", cubo.shape, "| soma de n =", cubo["n"].sum(), "(deve = nº de movimentações)")
assert cubo["n"].sum() == len(dfm)

# Escolaridade × tipo de movimento por mesorregião (perfil admitido × desligado, salário por grau)
esc_reg = (dfm.groupby(["mesorregiao","tipo","grau_cod","grau_instr"], dropna=False)
             .agg(n=("saldo","size"), sal_soma=("salario","sum"), sal_n=("salario","count"))
             .reset_index())

# Série mensal (mês do fato, restrita à janela de declaração set/2025–ago/2026)
serie = (dfm[dfm["competênciamov"].between("202509", "202608")]
           .groupby(["competênciamov","mesorregiao","macrofamilia"])
           .agg(admissoes=("admissao","sum"), desligamentos=("desligamento","sum"),
                saldo=("saldo","sum"))
           .reset_index())

saidas = {
    "escolaridade_regiao.parquet":  esc_reg,
    "serie_mensal.parquet":         serie,
    "agg_municipio.parquet":        terr,
    "agg_mesorregiao.parquet":      reg,
    "agg_setor.parquet":            setor,
    "agg_municipio_setor.parquet":  terr_setor,
    "perfil_adm_desl.parquet":      perfil,
    "salario_por_grau.parquet":     sal_grau,
    "macrofamilias_cbo.parquet":    macro,
    "macrofamilia_mesorregiao.parquet": macro_reg,
    "cubo_movimentacoes.parquet":   cubo,
    "municipios.parquet":           mun,
}
for nome, tabela in saidas.items():
    caminho = os.path.join(OUTPUTS, nome)
    tabela.to_parquet(caminho, index=False)
    print(f"gravado: {caminho}  ({tabela.shape[0]} linhas x {tabela.shape[1]} cols)")

print("\nPipeline concluído. Tabelas prontas para o dashboard.")

Cubo: (57443, 14) | soma de n = 3397416 (deve = nº de movimentações)


gravado: ../outputs/escolaridade_regiao.parquet  (144 linhas x 7 cols)
gravado: ../outputs/serie_mensal.parquet  (1063 linhas x 6 cols)
gravado: ../outputs/agg_municipio.parquet  (295 linhas x 19 cols)
gravado: ../outputs/agg_mesorregiao.parquet  (6 linhas x 16 cols)
gravado: ../outputs/agg_setor.parquet  (22 linhas x 14 cols)
gravado: ../outputs/agg_municipio_setor.parquet  (4425 linhas x 17 cols)
gravado: ../outputs/perfil_adm_desl.parquet  (24 linhas x 4 cols)
gravado: ../outputs/salario_por_grau.parquet  (12 linhas x 4 cols)
gravado: ../outputs/macrofamilias_cbo.parquet  (15 linhas x 14 cols)
gravado: ../outputs/macrofamilia_mesorregiao.parquet  (90 linhas x 17 cols)
gravado: ../outputs/cubo_movimentacoes.parquet  (57443 linhas x 14 cols)
gravado: ../outputs/municipios.parquet  (295 linhas x 9 cols)

Pipeline concluído. Tabelas prontas para o dashboard.


## 10. Como ler os resultados (resumo analítico)

- **Expansão acelerada**: municípios/setores com `saldo_por_10k` alto e crescente.
- **Apagão de mão de obra**: `giro_por_10k` alto com `saldo` baixo/negativo — reposição
  constante sem crescimento líquido.
- **Causa-raiz (renda/escolaridade)**:
  - salário setorial abaixo da média + saldo negativo → **desatratividade salarial**;
  - vagas exigindo grau alto em região de baixa `escolarizacao`/IDHM → **escolaridade não atendida**;
  - salário estável apesar de maior exigência de instrução → **compressão salarial**.
- **IA/CBO**: as macrofamílias geradas pelo LLM mostram se o gargalo é **operacional**
  (e de qual cadeia: metalmecânica, têxtil, alimentos...) ou **técnico/engenharia** —
  base direta para as recomendações do pitch.

> **Limites da análise:** saldo e giro usam o próprio CAGED (fluxo, não estoque de vínculos);
> exclusões (arquivo EXC) não foram fornecidas; IDHM é de 2010; salários fora de
> R$ 500–50.000 foram ignorados nas médias; anos de estudo são aproximação do grau de
> instrução; promoções internas não aparecem no CAGED (o saldo negativo de Gestão/supervisão
> pode refletir vagas preenchidas internamente, não só cortes); a classificação de CBOs por LLM pode errar em ocupações ambíguas (validada por
> coerência com a CBO oficial e por amostra revisada pela equipe).
